In [ ]:
# Sequential vs Tree-Guided Comparison (51-100)

In [25]:
import json
from pathlib import Path

import pandas as pd

benchmark = pd.read_csv('../data/51_to_100_benchmark.csv')
benchmark = benchmark.rename(columns={'Value': 'benchmark_risk'})
benchmark['paper_id'] = benchmark['paper_id'].astype(str).str.zfill(3)
benchmark['domain'] = benchmark['Domain'].str.replace('Domain ', 'domain_')
benchmark['benchmark_risk'] = benchmark['benchmark_risk'].str.strip().replace(
    {'Not applicable': 'Not Applicable'}
)

with Path('../output/sequential_decision_tree_all_papers/51_to_100/all_papers_anthropic_claude-opus-5.json').open() as file:
    sequential_data = json.load(file)
with Path('../output/all_papers_all_domains_two_prompts/51_to_100/tree_guided_all_papers_anthropic_claude-opus-5.json').open() as file:
    tree_guided_data = json.load(file)

def risk_label(value):
    value = str(value).upper()
    if 'NOT APPLICABLE' in value:
        return 'Not Applicable'
    if 'HIGH' in value:
        return 'High'
    if 'MEDIUM' in value or 'MED' in value:
        return 'Medium'
    if 'LOW' in value:
        return 'Low'

sequential_rows = []
for paper_id, domains in sequential_data.items():
    for domain, result_data in domains.items():
        if domain.isdigit():
            result = result_data.get('result') if isinstance(result_data, dict) else result_data
            sequential_rows.append({
                'paper_id': paper_id,
                'domain': f'domain_{domain}',
                'sequential_risk': risk_label(result),
            })

tree_guided_rows = []
for paper_file, domains in tree_guided_data['results'].items():
    for domain, result_data in domains.items():
        judgement = result_data.get('parsed_response', {}).get('final_risk_judgement')
        tree_guided_rows.append({
            'paper_id': paper_file.removesuffix('.json'),
            'domain': domain,
            'tree_guided_risk': risk_label(judgement),
        })

comparison = (
    benchmark[['paper_id', 'domain', 'benchmark_risk']]
    .merge(pd.DataFrame(sequential_rows), on=['paper_id', 'domain'])
    .merge(pd.DataFrame(tree_guided_rows), on=['paper_id', 'domain'])
    .dropna()
)
comparison['sequential_match'] = comparison['benchmark_risk'] == comparison['sequential_risk']
comparison['tree_guided_match'] = comparison['benchmark_risk'] == comparison['tree_guided_risk']

print(f'Compared: {len(comparison)}')
print(f"Sequential accuracy: {comparison['sequential_match'].mean():.1%}")
print(f"Tree-guided accuracy: {comparison['tree_guided_match'].mean():.1%}")

display(comparison[~comparison['sequential_match'] | ~comparison['tree_guided_match']])

Compared: 332
Sequential accuracy: 56.9%
Tree-guided accuracy: 50.3%


,paper_id,domain,benchmark_risk,sequential_risk,tree_guided_risk,sequential_match,tree_guided_match
1,051,domain_2,Low,High,High,False,False
2,051,domain_3,Low,Not Applicable,Low,False,True
3,051,domain_4,Not Applicable,High,High,False,False
5,051,domain_6,Low,High,Medium,False,False
6,051,domain_7,Medium,High,High,False,False
...,...,...,...,...,...,...,...
328,099,domain_7,Medium,High,High,False,False
329,100,domain_1,Low,High,High,False,False
330,100,domain_2,Medium,High,Medium,False,True
334,100,domain_6,Medium,High,Medium,False,True
